# 01 · Data preparation - from raw documents to traceable knowledge records

**Objective.** Turn the raw source documents into clean, traceable knowledge records (JSONL), and only from sources that
passed the activation gate (licence, scope, file integrity, optional clinical approval). Then build the FAISS + BM25 indexes.

| Stage | What happens | Output |
|---|---|---|
| 1 | Read the curated source registry | `data/source_registry.csv` |
| 2 | Apply the activation gate | `results/data_quality/source_gate_report.json` |
| 3 | Extract text from every active source | `data/processed/pages/`, `data/processed/medications/` |
| 4 | Parse the medication PDFs (Saudi KB, StatPearls) | knowledge units with page numbers |
| 5 | Build knowledge records + deduplicate | `data/processed/knowledge/<collection>.jsonl` |
| 6 | Inspect the record schema | - |
| 7 | Review queues for the medical team | `results/data_quality/*.csv` |
| 8 | Trace a chunk back to its document and page | - |
| 9 | Build and verify indexes (optional, ~3 min on CPU) | `indexes/<collection>/` |

All logic lives in `src/vitala_rag/knowledge.py`; this notebook calls it step by step and shows the intermediate results.
**Expected result:** 9 active sources, 4 inactive (ADA: licence; amlodipine / atorvastatin: out of scope; app screens: reference only).

In [ ]:
# Stage 0 - setup: find the project root, make `src/` importable, load .env (keys are never printed)
import sys, os, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "vitala_rag").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the VITALA_RAG project (could not find src/vitala_rag).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from vitala_rag.config import Paths
paths = Paths(root=ROOT)
print("Project root:", ROOT)
print("LLM provider configured:", os.environ.get("VITALA_LLM_PROVIDER", "(not set)"), "| API keys are not displayed")

## Stage 1 - the source registry

`data/source_registry.csv` is the **single, human-editable record** of every source. Important columns:

* `licence_status`: `permitted`, `permitted_internal_use`, `restricted` - only the first two may be indexed
* `scope_status`: `in_scope`, `out_of_scope`, `excluded`, `reference_only`
* `clinical_review_status`: `NOT_REVIEWED` / `APPROVED` / `REJECTED` (filled in by the medical team)
* `sha256`: the registered file hash - a replaced or edited file is blocked until someone re-reviews it
* `redistribution_allowed`: decides what the public GitHub export may contain

In [ ]:
from vitala_rag import knowledge as K
registry = pd.DataFrame(K.read_registry(paths.registry_csv))
registry[["source_id", "document_type", "collection", "licence_status", "scope_status", "clinical_review_status",
          "redistribution_allowed", "version"]]

## Stage 2 - the activation gate

Every check is listed, so a blocked source always says *why*. Set the environment variable
`VITALA_REQUIRE_CLINICAL_APPROVAL=1` to require clinician approval (production setting): with the current registry that
would block every source, because none has been clinically reviewed yet.

In [ ]:
gate = K.run_source_verification(paths)
pd.DataFrame([{"source_id": d["source_id"], "active": d["active"], "reasons": "; ".join(d["reasons"]),
               **{k: v for k, v in d["checks"].items()}} for d in gate["sources"]])

## Stage 3 - extraction

* Guideline PDFs: blocks with font sizes, tables as Markdown, repaired font encoding (see `pdf_extract.py`)
* FDA label: structured record from the raw openFDA JSON (offline; no network needed)
* Medication PDFs: plain page text is stored for traceability; their structure is parsed in stage 4

The review queue lists pages with tables, repaired glyphs or many numeric thresholds that a clinician should check.

In [ ]:
ext = K.run_extraction(paths)
print("pages to review:", ext["review_pages"])
pd.DataFrame(ext["documents"])[["source_id", "document_type"]]

## Stage 4 - parsing the medication PDFs

**Saudi medication knowledge base.** One record per `KSA-MED-NNNN` (the document's own stable identifier), with its
numbered section, page range, cited sources (`S001...`, resolved to URLs from the document's source registry) and
borderless tables rebuilt from cell positions.

**StatPearls reviews.** Paragraphs keep their section, sub-section and PDF page; reference markers like `[12]` are
removed; paragraphs broken by a page break are re-joined.

In [ ]:
medkb = paths.root / registry.set_index("source_id").loc["SA-MEDKB-2026", "raw_path"]
units, rep = K.parse_saudi_medkb(medkb)
print(rep)
example = next(u for u in units if u["record_id"] == "KSA-MED-0050")
print(json.dumps({k: example[k] for k in ("record_id", "title", "section_title", "page_start", "page_end", "references")}, indent=1)[:800])
print(example["text"])

In [ ]:
sp = paths.root / registry.set_index("source_id").loc["STATPEARLS-ACEI-2025", "raw_path"]
paras, rep = K.parse_statpearls(sp)
rep.pop("_tables")
print({k: v for k, v in rep.items() if k not in ("duplicate_paragraphs_dropped", "near_duplicate_paragraphs")})
pd.DataFrame(paras)[["h2", "h3", "page", "text"]].head(8)

## Stage 5 - knowledge records

Every chunk from every collection is normalised into **one record schema** (stage 6). Two policies apply here:

* **Deduplication.** Exact duplicates (same normalised-text hash) inside a collection are dropped; near-duplicates
  (word 5-gram Jaccard >= 0.9) are reported, not removed.
* **Dosing policy.** Dose-titration / maximum-dose references are kept in the JSONL with `active=false` and are not
  indexed (`VITALA_INDEX_DOSING=1` changes this). The chatbot must not be pulled into dosing advice.

In [ ]:
manifest = K.build_knowledge_base(paths)
print("dedup:", manifest["dedup"])
pd.DataFrame(manifest["collections"]).T[["records", "active_records", "sources"]]

In [ ]:
recs = pd.DataFrame([r for c in ("preventive", "medication", "app") for r in K.load_knowledge(paths, c, active_only=False)])
recs.groupby(["collection", "source_id", "content_type", "active"]).size().rename("records").to_frame()

## Stage 6 - the record schema

Look at one record: identity (`chunk_id`, `source_id`, `collection`), classification (`topic`, `medications`), provenance
(`url`, `raw_path`, `version`, dates, `pdf_page_start/end`, `section_path`), governance (`licence_status`,
`clinical_review_status`, `active`), integrity (`sha256` of the document, `content_sha256` of the chunk) and the text
used for display (`text`) versus for embedding (`text_for_embedding`).

In [ ]:
rec = recs.set_index("chunk_id").loc["SA-MEDKB-2026::KSA-MED-0098"]
rec.drop(["text_for_embedding"]).to_frame("value")

## Stage 7 - review queues for the medical team (open the CSV files in Excel)

In [ ]:
queue = pd.read_csv(paths.reports / "chunk_review_queue.csv")
print(len(queue), "chunks to review ->", paths.reports / "chunk_review_queue.csv")
queue[queue["known_issues"].fillna("").str.len() > 0][["chunk_id", "active", "known_issues"]].head(10)

## Stage 8 - traceability

`trace_chunk` follows any chunk back to the registry entry, the raw file, the cited page(s) and URL, checks the document
hash, and probes that phrases of the chunk really occur on the cited PDF page.

In [ ]:
for cid in ["SA-MEDKB-2026::KSA-MED-0075", "STATPEARLS-METFORMIN-2023::mechanism_of_action::02", "SA-PHA-SCPG-2023::p027::0061"]:
    t = K.trace_chunk(paths, cid)
    print(json.dumps({k: t[k] for k in ("chunk_id", "raw_path", "pdf_page_start", "url", "doc_hash_matches",
                                        "text_found_on_cited_pages", "clinical_review_status")}, ensure_ascii=False))

## Stage 9 - build and verify the indexes

Set `REBUILD_INDEXES = True` to re-embed every active record with `intfloat/multilingual-e5-base` (CPU: about 3 minutes;
the model is downloaded once). Indexes are deterministic for the same records: exact FAISS (`IndexFlatIP`), and the
manifest stores the hashes of the knowledge file and of every source document.

In [ ]:
REBUILD_INDEXES = False   # True: rebuild indexes/preventive, indexes/medication, indexes/app

if REBUILD_INDEXES:
    from vitala_rag.retrieval import Embedder
    K.build_indexes(paths, Embedder())

from vitala_rag.retrieval import verify_index
pd.DataFrame([verify_index(paths.indexes / c) for c in ("preventive", "medication", "app")])

**Expected result.** Three indexes with `ok=True`, and `n_chunks` equal to the number of *active* records per
collection. If a row says the hash does not match, the knowledge records changed after the last index build: set
`REBUILD_INDEXES = True` and run stage 9 again.